# Ahmed Twin — Evaluation: SFT vs. DPO

Compares the plain SFT model against the final DPO model (`dpo_v7_rpo`, chosen in the DPO stage) on real conversations neither model was trained on.

### Three independent kinds of evidence
1. **Likelihood of Ahmed's real reply (objective).** For each held-out conversation: which model assigns higher probability to what Ahmed *actually* said? This needs no judge and has no opinions.
2. **Automatic checks on the generated replies.** Length, rambling (hitting the token limit), generic-assistant phrases, wrong dialect, private-name leaks, and character overlap with the real reply.
3. **LLM-as-judge (optional).** An external model sees the conversation, the real reply, and both candidates (unlabeled) and picks which sounds more like the real person. Each pair is judged **in both orders**, so position bias cancels out.

Every comparison is **paired** (same conversations for both models) and comes with a significance test. "Better" means distinguishable from noise.

### What changed from the previous version
| Previous problem | Now |
|---|---|
| Evaluated the old v6 adapter | Evaluates `runs/dpo_v7_rpo/adapter`, the run selected in the DPO stage, and verifies it by fingerprint |
| Cerebras key hardcoded (and the "is it set?" check compared against a different key) | Read from Colab Secrets or entered at runtime; never stored in the notebook |
| Identical SFT/DPO replies were sent to the judge, producing pure coin flips | Identical replies count as ties and are never judged |
| Resume keyed by the last user message, so different conversations ending in "تمام" collided | Resume keyed by a unique record id, with a config fingerprint so settings can't silently change mid-run |
| Final table lost the verdicts loaded from disk after a resume | The report is rebuilt from the saved files, never from memory |
| `"CANDIDATE B"` could be parsed as "A" | Strict parsing: exactly one of A / B / TIE, otherwise "no answer" |
| One random order per pair; seed meaningless across threads | Both orders judged; a win needs both to agree |
| The promised rate limit didn't exist; errors were swallowed | A real shared rate limiter; errors are logged and counted |
| `temperature` silently ignored (no `do_sample`) | Explicit greedy decoding: deterministic and reproducible |
| "Win rate > 55%" with no significance test | Exact paired sign tests and confidence intervals |
| Two full model copies in memory | One copy: all SFT replies first, then the adapter is attached for DPO |
| The judge saw only the last message | The judge sees the last few turns |
| Private chats sent to a third-party API without notice | Explicit opt-in, plus name anonymization before sending |

### Before running
- Train/eval data: `ahmed_twin/validation.jsonl` (unseen by both models' training).
- `ahmed_twin/private_names.txt`: real names, one per line (`#` lines are comments). It's used for leak detection and judge anonymization, with whole-word matching, so a short name never matches inside a longer word. Keep it on Drive only; it's in `.gitignore`.
- Judge provider: Cerebras is the default; Groq (free tier, same `gpt-oss-120b` model) is a one-line switch in A2. Add the key in Colab's 🔑 Secrets panel under `CEREBRAS_API_KEY` (or `GROQ_API_KEY`), or you'll be prompted.
- **Before committing: Edit → Clear all outputs.** Outputs contain real messages.

---
## Part A — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q unsloth openai

### A1. Imports and GPU check
`unsloth` is imported first so its patches apply. `UNSLOTH_RETURN_LOGITS=1` is needed for the likelihood metric.

In [ ]:
import os
os.environ["UNSLOTH_RETURN_LOGITS"] = "1"

import unsloth  # must come before transformers / peft
from unsloth import FastLanguageModel

import json, re, time, math, random, hashlib, shutil, threading
import numpy as np
import torch
from transformers import set_seed

assert torch.cuda.is_available(), "No GPU detected -- Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

### A2. Configuration — the only cell you should need to edit

In [ ]:
# ======================= EVAL CONFIG =======================
EVAL_RUN_NAME = "ch7_sft_vs_dpo_v7"   # one folder per evaluation run
START_FRESH = False                   # True = delete this run's saved progress and start over

DRIVE_ROOT = "/content/drive/MyDrive"
SFT_MODEL_PATH = f"{DRIVE_ROOT}/ahmed_twin/model_merged_nilechat"     # SFT model (checkpoint-1000, merged)
DPO_ADAPTER_PATH = f"{DRIVE_ROOT}/DPO/runs/dpo_v7_rpo/adapter"         # final DPO model, selected in the DPO stage
HELD_OUT_PATH = f"{DRIVE_ROOT}/ahmed_twin/validation.jsonl"            # unseen by SFT and DPO training
DPO_DEV_PATH = f"{DRIVE_ROOT}/DPO/dpo_eval_hard.jsonl"                 # used to pick the DPO checkpoint -> excluded here
PRIVATE_NAMES_PATH = f"{DRIVE_ROOT}/ahmed_twin/private_names.txt"      # optional; real names, one per line

EVAL_DIR = f"{DRIVE_ROOT}/DPO/eval_runs/{EVAL_RUN_NAME}"
SFT_OUTPUTS_PATH = f"{EVAL_DIR}/sft_outputs.jsonl"
DPO_OUTPUTS_PATH = f"{EVAL_DIR}/dpo_outputs.jsonl"
SAMPLE_PATH = f"{EVAL_DIR}/eval_sample.jsonl"
CONFIG_PATH = f"{EVAL_DIR}/eval_config.json"
SUMMARY_PATH = f"{EVAL_DIR}/eval_summary.json"

TABLE_PATH = f"{EVAL_DIR}/eval_table.csv"

SEED = 0
SAMPLE_SEED = 2026            # which held-out conversations get sampled
NUM_EVAL_SAMPLES = 150        # ~20-25 min of generation on a T4 for both models
MAX_SEQ_LENGTH = 2048
MAX_CONTEXT_TOKENS = 1024     # longer contexts drop their oldest turns (system message always kept)
GENERATION = dict(max_new_tokens=150, do_sample=False, repetition_penalty=1.1, no_repeat_ngram_size=4)

RUN_LLM_JUDGE = True          # sends conversation excerpts to an external API -- see Part E
# Any OpenAI-compatible provider works. Presets (keep exactly one line uncommented):
#   Cerebras:
JUDGE_BASE_URL, JUDGE_MODEL, JUDGE_KEY_NAME = "https://api.cerebras.ai/v1", "gpt-oss-120b", "CEREBRAS_API_KEY"
#   Groq (free tier, same gpt-oss-120b model):
# JUDGE_BASE_URL, JUDGE_MODEL, JUDGE_KEY_NAME = "https://api.groq.com/openai/v1", "openai/gpt-oss-120b", "GROQ_API_KEY"
JUDGE_REQUESTS_PER_MINUTE = 25   # free tiers allow ~30/min; stay below it
JUDGE_WORKERS = 3
JUDGE_CONTEXT_TURNS = 4          # how many recent turns the judge sees
ANONYMIZE_FOR_JUDGE = True       # replace names from private_names.txt before sending
JUDGE_PROMPT_VERSION = 2         # bump if you edit the judge prompt (invalidates old judgments)

GENERIC_AI_PHRASES = ["كذكاء اصطناعي", "ذكاء اصطناعي", "ماعنديش مشاعر", "ما عنديش مشاعر",
                      "أنا هنا عشان أساعدك", "أنا هنا باش نساعدك", "لو عندك أي سؤال",
                      "متترددش تسألني", "العفو! أنا هنا", "كيف يمكنني مساعدتك"]
DARIJA_MARKERS = ["شنو", "بغيتي", "ديالي", "واخا", "بزاف"]

set_seed(SEED)

### A3. Run folder, config fingerprint, private names

Saved progress is only reused if every setting that affects results is unchanged. Changing the model, sample, decoding, or judge prompt without `START_FRESH = True` stops the run, instead of silently mixing old and new results.

In [ ]:
# Fingerprint = settings that affect the GENERATED REPLIES. Judge settings are kept out on purpose:
# changing or switching the judge must never force re-generating replies.
CONFIG = json.loads(json.dumps({
    "sft_model": SFT_MODEL_PATH, "dpo_adapter": DPO_ADAPTER_PATH, "held_out": HELD_OUT_PATH,
    "sample_seed": SAMPLE_SEED, "num_samples": NUM_EVAL_SAMPLES, "max_context_tokens": MAX_CONTEXT_TOKENS,
    "generation": GENERATION,
}))
# Each judge (model + prompt version + context/anonymization settings) gets its own verdict file.
JUDGE_TAG = f"{re.sub(r'[^A-Za-z0-9]+', '-', JUDGE_MODEL)}_p{JUDGE_PROMPT_VERSION}_c{JUDGE_CONTEXT_TURNS}_a{int(ANONYMIZE_FOR_JUDGE)}"
JUDGMENTS_PATH = f"{EVAL_DIR}/judgments_{JUDGE_TAG}.jsonl"

if START_FRESH and os.path.isdir(EVAL_DIR):
    shutil.rmtree(EVAL_DIR)
    print("START_FRESH: deleted", EVAL_DIR)
os.makedirs(EVAL_DIR, exist_ok=True)

if os.path.exists(CONFIG_PATH):
    saved = json.load(open(CONFIG_PATH, encoding="utf-8"))
    changed = sorted(k for k in CONFIG if saved.get(k) != CONFIG[k])   # older runs may hold extra keys: ignored
    if changed:
        raise RuntimeError(
            f"Settings changed since this eval run started: {changed}. Use a new EVAL_RUN_NAME, "
            "or set START_FRESH = True to discard the saved progress."
        )
    print("Resuming eval run:", EVAL_DIR)
else:
    json.dump(CONFIG, open(CONFIG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("New eval run:", EVAL_DIR)

# Same matching rules as tools/private_names_lib.py, used across the whole project.
ARABIC = "\u0600-\u06FF"
NAME_PREFIXES = "وبل"   # و/ب/ل attach to names. NOT ف or ك: ف/ك + a short name often spells an everyday word

PRIVATE_NAMES = []
if os.path.exists(PRIVATE_NAMES_PATH):
    PRIVATE_NAMES = sorted({l.strip() for l in open(PRIVATE_NAMES_PATH, encoding="utf-8")
                            if l.strip() and not l.strip().startswith("#")}, key=len, reverse=True)

_ar_names = [n for n in PRIVATE_NAMES if re.search(f"[{ARABIC}]", n)]
_la_names = [n for n in PRIVATE_NAMES if n not in _ar_names]
# Whole-word matches only: a short name must never match inside a longer everyday word.
NAME_RE_AR = (re.compile(rf"(?<![{ARABIC}])(?P<pre>[{NAME_PREFIXES}])?(?P<name>{'|'.join(map(re.escape, _ar_names))})(?![{ARABIC}])")
              if _ar_names else None)
NAME_RE_LA = (re.compile(rf"(?<![A-Za-z])(?:{'|'.join(map(re.escape, _la_names))})(?![A-Za-z])", re.IGNORECASE)
              if _la_names else None)


def mentions_private_name(text):
    return bool((NAME_RE_AR and NAME_RE_AR.search(text)) or (NAME_RE_LA and NAME_RE_LA.search(text)))


print(f"Private names loaded: {len(PRIVATE_NAMES)}" + ("" if PRIVATE_NAMES else
      "  (no private_names.txt -> name-leak check and judge anonymization are off)"))

### A4. Judge API key (only if the judge is enabled)

Read from Colab Secrets (🔑 panel, under the name in `JUDGE_KEY_NAME`, e.g. `GROQ_API_KEY`), otherwise prompted. Never written to the notebook. **An old Cerebras key was committed in a previous version of this notebook. Revoke it in the Cerebras dashboard if you haven't already.**

In [ ]:
JUDGE_API_KEY = None
if RUN_LLM_JUDGE:
    try:
        from google.colab import userdata
        JUDGE_API_KEY = userdata.get(JUDGE_KEY_NAME)
    except Exception:
        pass
    if not JUDGE_API_KEY:
        from getpass import getpass
        JUDGE_API_KEY = getpass(f"{JUDGE_KEY_NAME}: ").strip()
    assert JUDGE_API_KEY, f"No {JUDGE_KEY_NAME} -- set RUN_LLM_JUDGE = False to skip the judge."
    print(f"{JUDGE_KEY_NAME} loaded (not displayed). Judge: {JUDGE_MODEL} @ {JUDGE_BASE_URL}")
else:
    print("LLM judge disabled -- Parts C, D and F still run.")

---
## Part B — Model and held-out data

### B1. Load the SFT model

Only one copy of the model is loaded. Part C generates every SFT reply first, then attaches the DPO adapter to the same model. That avoids both the memory cost of two copies and any adapter on/off toggling.

In [ ]:
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL_PATH, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=False,
)


def resolve_eos_id(tok, preferred="<|im_end|>"):
    token_id = tok.convert_tokens_to_ids(preferred)
    return tok.eos_token_id if token_id is None or token_id == tok.unk_token_id else token_id


EOS_ID = resolve_eos_id(tokenizer)
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else EOS_ID
assert getattr(tokenizer, "chat_template", None), "Tokenizer has no chat template."
print("Loaded SFT model:", SFT_MODEL_PATH, "| end-of-turn id:", EOS_ID)

### B2. Build the held-out evaluation sample

- **Source:** `validation.jsonl`, never used to train SFT or DPO.
- **Sliding-window duplicates removed:** the same exchange appears in several overlapping windows; one copy is kept.
- **DPO dev conversations excluded:** they were used to pick the DPO checkpoint, so they aren't fully neutral for comparing SFT and DPO.
- **Stable ids:** each conversation gets an id, so resuming can never mix up two conversations.
- **Frozen to disk:** the sample is saved, so a resumed run evaluates exactly the same conversations.

In [ ]:
def record_id(rec):
    return rec.get("id") or hashlib.sha1(json.dumps(rec["messages"], ensure_ascii=False).encode()).hexdigest()[:20]


def prompt_for(messages):
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


if os.path.exists(SAMPLE_PATH):
    eval_records = [json.loads(l) for l in open(SAMPLE_PATH, encoding="utf-8") if l.strip()]
    print(f"Loaded frozen sample of {len(eval_records)} conversations from {SAMPLE_PATH}")
else:
    raw = [json.loads(l) for l in open(HELD_OUT_PATH, encoding="utf-8") if l.strip()]
    usable = [r for r in raw
              if len(r.get("messages", [])) >= 2
              and r["messages"][-1]["role"] == "assistant"
              and r["messages"][-1]["content"].strip()
              and any(m["role"] == "user" for m in r["messages"][:-1])]

    unique = {}
    for r in usable:
        last_user = next(m["content"] for m in reversed(r["messages"][:-1]) if m["role"] == "user")
        unique[(last_user.strip(), r["messages"][-1]["content"].strip())] = r   # keep the longest-context window

    dev_prompts = set()
    if os.path.exists(DPO_DEV_PATH):
        dev_prompts = {json.loads(l)["prompt"] for l in open(DPO_DEV_PATH, encoding="utf-8") if l.strip()}
    candidates = [r for r in unique.values() if prompt_for(r["messages"][:-1]) not in dev_prompts]

    print(f"validation.jsonl: {len(raw)} rows -> {len(usable)} usable -> {len(unique)} unique exchanges "
          f"-> {len(candidates)} after excluding {len(unique) - len(candidates)} DPO-dev conversations")

    candidates.sort(key=record_id)                    # order-independent sampling
    rng = random.Random(SAMPLE_SEED)
    eval_records = rng.sample(candidates, min(NUM_EVAL_SAMPLES, len(candidates)))
    for r in eval_records:
        r["id"] = record_id(r)
    assert len({r["id"] for r in eval_records}) == len(eval_records), "Duplicate ids in sample."
    with open(SAMPLE_PATH, "w", encoding="utf-8") as f:
        for r in eval_records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"Sampled and froze {len(eval_records)} conversations -> {SAMPLE_PATH}")

RECORDS_BY_ID = {r["id"]: r for r in eval_records}

---
## Part C — Generate and score (SFT first, then DPO)

### C1. Helpers

- **Context fitting:** prompts longer than `MAX_CONTEXT_TOKENS` drop their oldest turns. The system message and the latest turns are always kept.
- **Generation:** greedy (deterministic). It also records whether the reply hit the token limit, which flags rambling.
- **Likelihood of the real reply:** average negative log-likelihood per token of what Ahmed actually wrote, including the end-of-turn token, so "knowing when to stop" counts too. **Lower is better.**

In [ ]:
def fit_context(messages):
    msgs = list(messages)
    while True:
        prompt = prompt_for(msgs)
        n = len(tokenizer(prompt, add_special_tokens=False).input_ids)
        non_system = [i for i, m in enumerate(msgs) if m["role"] != "system"]
        if n <= MAX_CONTEXT_TOKENS or len(non_system) <= 1:
            return prompt, n
        del msgs[non_system[0]]          # drop the oldest non-system turn


@torch.no_grad()
def generate_reply(prompt_text):
    enc = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**enc, **GENERATION, eos_token_id=EOS_ID, pad_token_id=PAD_ID)
    new_ids = out[0, enc["input_ids"].shape[1]:]
    hit_limit = bool(len(new_ids) >= GENERATION["max_new_tokens"] and EOS_ID not in new_ids.tolist())
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip(), int(len(new_ids)), hit_limit


@torch.no_grad()
def real_reply_nll(prompt_text, reply):
    p_ids = tokenizer(prompt_text, add_special_tokens=False).input_ids
    c_ids = tokenizer(reply, add_special_tokens=False).input_ids + [EOS_ID]
    ids = torch.tensor([p_ids + c_ids], device="cuda")
    logits = model(input_ids=ids).logits[0, :-1].float()
    token_lp = torch.log_softmax(logits, dim=-1).gather(1, ids[0, 1:, None]).squeeze(1)
    return float(-token_lp[-len(c_ids):].mean().item())


def run_phase(label, out_path):
    done = set()
    if os.path.exists(out_path):
        done = {json.loads(l)["id"] for l in open(out_path, encoding="utf-8") if l.strip()}
    todo = [r for r in eval_records if r["id"] not in done]
    print(f"{label}: {len(done)} already done, {len(todo)} to go")
    FastLanguageModel.for_inference(model)
    from tqdm.auto import tqdm
    with open(out_path, "a", encoding="utf-8") as f:
        for r in tqdm(todo, desc=f"{label} replies"):
            prompt, n_ctx = fit_context(r["messages"][:-1])
            reply, n_new, hit_limit = generate_reply(prompt)
            nll = real_reply_nll(prompt, r["messages"][-1]["content"])
            f.write(json.dumps({"id": r["id"], "reply": reply, "new_tokens": n_new, "hit_limit": hit_limit,
                                "real_reply_nll": nll, "context_tokens": n_ctx}, ensure_ascii=False) + "\n")
            f.flush()


def load_outputs(path):
    rows = {}
    if os.path.exists(path):
        for l in open(path, encoding="utf-8"):
            if l.strip():
                row = json.loads(l)
                rows[row["id"]] = row          # last write wins (safe if a line was written twice)
    return rows

### C2. SFT phase (no adapter attached)

Resumable: after a disconnect, re-run Part A, then B, then this cell. It continues where it stopped.

In [ ]:
assert not isinstance(model, PeftModel), (
    "The DPO adapter is already attached -- SFT replies must come from the plain model. "
    "Restart the runtime and run Parts A-B again before this cell."
)
run_phase("SFT", SFT_OUTPUTS_PATH)
assert len(load_outputs(SFT_OUTPUTS_PATH)) == len(eval_records), "SFT phase incomplete."

### C3. Attach the DPO adapter and prove it's the right one

Three checks before any DPO reply is generated:
1. **Right file:** the adapter loaded into the model matches the file on disk (fingerprint).
2. **Right run:** the fingerprint matches the one the DPO stage recorded for v7's best checkpoint (`run_summary.json`).
3. **Actually active:** a probe prompt's output changes once the adapter is attached.

In [ ]:
from safetensors.torch import load_file


def lora_fp(m):
    return sum(p.detach().float().abs().sum().item() for n, p in m.named_parameters() if "lora_" in n)


probe_prompt = prompt_for([{"role": "system", "content": eval_records[0]["messages"][0]["content"]},
                           {"role": "user", "content": "ازيك عامل ايه النهارده؟"}])
probe_ids = tokenizer(probe_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to("cuda")
FastLanguageModel.for_inference(model)
with torch.no_grad():
    sft_probe = model(input_ids=probe_ids).logits[0, -1].float().cpu()

if not isinstance(model, PeftModel):
    model = PeftModel.from_pretrained(model, DPO_ADAPTER_PATH)
model.eval()

file_state = load_file(os.path.join(DPO_ADAPTER_PATH, "adapter_model.safetensors"))
file_fp = sum(t.float().abs().sum().item() for k, t in file_state.items() if "lora_" in k)
loaded_fp = lora_fp(model)
assert abs(loaded_fp - file_fp) / file_fp < 1e-3, f"Adapter mismatch: model {loaded_fp:.2f} vs file {file_fp:.2f}"
print(f"1. Adapter in model matches the file on disk ({loaded_fp:.2f}).")

summary_path = os.path.join(os.path.dirname(DPO_ADAPTER_PATH), "run_summary.json")
if os.path.exists(summary_path):
    recorded = json.load(open(summary_path, encoding="utf-8"))["lora_fingerprint"]["after"]
    assert abs(loaded_fp - recorded) / recorded < 1e-3, (
        f"Adapter fingerprint {loaded_fp:.2f} != {recorded:.2f} recorded by the DPO stage for this run. "
        "Wrong adapter folder?")
    print(f"2. Matches the fingerprint the DPO stage recorded for this run ({recorded:.2f}).")
else:
    print("2. (skipped) no run_summary.json next to the adapter to cross-check against.")

FastLanguageModel.for_inference(model)
with torch.no_grad():
    dpo_probe = model(input_ids=probe_ids).logits[0, -1].float().cpu()
shift = (dpo_probe - sft_probe).abs().max().item()
assert shift > 1e-3, "Adapter attached but the output didn't change -- it isn't active."
print(f"3. Adapter is active (probe logits moved by up to {shift:.3f}).")

### C4. DPO phase (same conversations, same decoding)

In [ ]:
assert isinstance(model, PeftModel), "Run C3 first -- the DPO adapter isn't attached."
run_phase("DPO", DPO_OUTPUTS_PATH)
assert len(load_outputs(DPO_OUTPUTS_PATH)) == len(eval_records), "DPO phase incomplete."

---
## Part D — Objective and automatic metrics

### D1. Merge everything into one table

Everything is rebuilt from the files on disk, so the numbers are identical whether the run happened in one go or across several sessions.

In [ ]:
import pandas as pd

sft_out, dpo_out = load_outputs(SFT_OUTPUTS_PATH), load_outputs(DPO_OUTPUTS_PATH)
rows = []
for r in eval_records:
    msgs = r["messages"]
    last_user = next(m["content"] for m in reversed(msgs[:-1]) if m["role"] == "user")
    rows.append({
        "id": r["id"], "last_user_msg": last_user, "real": msgs[-1]["content"],
        "sft": sft_out[r["id"]]["reply"], "dpo": dpo_out[r["id"]]["reply"],
        "sft_nll": sft_out[r["id"]]["real_reply_nll"], "dpo_nll": dpo_out[r["id"]]["real_reply_nll"],
        "sft_hit_limit": sft_out[r["id"]]["hit_limit"], "dpo_hit_limit": dpo_out[r["id"]]["hit_limit"],
    })
df = pd.DataFrame(rows)
print(f"{len(df)} conversations, both models complete.")

### D2. Statistics helpers

- **Sign test** (exact, two-sided): out of the conversations where the two models differ, does one win more often than chance?
- **Bootstrap 95% CI**: the range for the average difference.
- **Wilson 95% CI**: the range for a win rate.

In [ ]:
def sign_test(wins_a, wins_b):
    n = wins_a + wins_b
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(wins_a, wins_b) + 1)) / 2 ** n)


def bootstrap_ci(values, n_boot=5000, seed=SEED):
    values = np.asarray(values, dtype=float)
    rng_b = np.random.default_rng(seed)
    means = values[rng_b.integers(0, len(values), (n_boot, len(values)))].mean(axis=1)
    return float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 1.0)
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return (max(0.0, centre - half), min(1.0, centre + half))


def normalize(text):
    return re.sub(r"\s+", " ", text).strip()


def char_ngram_f1(candidate, reference, n=3):
    # chrF-style overlap: shared character n-grams (spaces removed), as an F1 score in [0, 1].
    def grams(t):
        t = re.sub(r"\s+", "", t)
        return [t[i:i + n] for i in range(len(t) - n + 1)] if len(t) >= n else ([t] if t else [])
    from collections import Counter
    c, r = Counter(grams(candidate)), Counter(grams(reference))
    overlap = sum((c & r).values())
    if not c or not r or overlap == 0:
        return 0.0
    precision, recall = overlap / sum(c.values()), overlap / sum(r.values())
    return 2 * precision * recall / (precision + recall)

### D3. Objective metric: how likely is Ahmed's real reply under each model?

This is the headline metric: no judge, no opinions. For every conversation, the model that assigns lower NLL (higher probability) to what Ahmed actually wrote "wins" that conversation.

In [ ]:
diff = df["dpo_nll"] - df["sft_nll"]            # negative = DPO makes the real reply MORE likely
dpo_better, sft_better = int((diff < 0).sum()), int((diff > 0).sum())
lo, hi = bootstrap_ci(diff)
nll_p = sign_test(dpo_better, sft_better)

print(f"Average NLL of the real reply (lower is better):  SFT {df['sft_nll'].mean():.3f}   DPO {df['dpo_nll'].mean():.3f}")
print(f"Average difference DPO - SFT: {diff.mean():+.4f}   95% CI [{lo:+.4f}, {hi:+.4f}]")
print(f"Conversations where DPO makes the real reply more likely: {dpo_better} / {len(df)}  "
      f"(SFT: {sft_better})   sign test p = {nll_p:.4f}")

NLL_RESULT = {"sft_mean": float(df["sft_nll"].mean()), "dpo_mean": float(df["dpo_nll"].mean()),
              "mean_diff": float(diff.mean()), "ci95": [lo, hi],
              "dpo_better": dpo_better, "sft_better": sft_better, "p_value": nll_p}

### D4. Automatic checks on the generated replies

Each column is compared with the **real replies** as the reference point. The goal is to look like Ahmed, not to be "short" or "long" in the abstract.

- **identical**: SFT and DPO produced exactly the same reply. These are ties for every comparison.
- **hit token limit**: the reply never ended on its own, a sign of rambling.
- **generic AI phrasing / Darija**: the failure patterns seen in earlier chapters.
- **private name**: mentions a name from `private_names.txt` (privacy check).
- **char overlap with real**: chrF-style character 3-gram F1 against Ahmed's actual reply. It's rough (two good replies can share few characters), but it's paired and free.

In [ ]:
def has_any(text, phrases):
    return any(p in text for p in phrases)


def describe(col):
    texts = df[col]
    out = {
        "avg chars": texts.str.len().mean(),
        "median chars": texts.str.len().median(),
        "empty": int((texts.str.strip() == "").sum()),
        "generic AI phrasing": int(texts.apply(lambda t: has_any(t, GENERIC_AI_PHRASES)).sum()),
        "Darija markers": int(texts.apply(lambda t: has_any(t, DARIJA_MARKERS)).sum()),
    }
    if PRIVATE_NAMES:
        out["mentions a private name"] = int(texts.apply(mentions_private_name).sum())
    if col != "real":
        out["hit token limit"] = int(df[f"{col}_hit_limit"].sum())
        out["avg char overlap with real"] = float(df.apply(lambda r: char_ngram_f1(r[col], r["real"]), axis=1).mean())
    return out


checks = pd.DataFrame({"REAL (Ahmed)": describe("real"), "SFT": describe("sft"), "DPO": describe("dpo")})
print(checks.round(3).to_string())

df["identical"] = df.apply(lambda r: normalize(r["sft"]) == normalize(r["dpo"]), axis=1)
print(f"\nIdentical SFT/DPO replies: {int(df['identical'].sum())} / {len(df)}")

df["sft_overlap"] = df.apply(lambda r: char_ngram_f1(r["sft"], r["real"]), axis=1)
df["dpo_overlap"] = df.apply(lambda r: char_ngram_f1(r["dpo"], r["real"]), axis=1)
ov_dpo = int((df["dpo_overlap"] > df["sft_overlap"]).sum())
ov_sft = int((df["sft_overlap"] > df["dpo_overlap"]).sum())
ov_p = sign_test(ov_dpo, ov_sft)
print(f"Closer to the real reply by character overlap: DPO {ov_dpo}, SFT {ov_sft}, ties {len(df) - ov_dpo - ov_sft}"
      f"   sign test p = {ov_p:.4f}")

OVERLAP_RESULT = {"dpo_closer": ov_dpo, "sft_closer": ov_sft, "p_value": ov_p}
CHECKS = json.loads(checks.to_json())

---
## Part E — LLM-as-judge (optional)

**Privacy note.** This part sends excerpts of real private conversations (the last few turns, Ahmed's real reply, and both model replies) to a third-party API (the provider set in `JUDGE_BASE_URL`). With `ANONYMIZE_FOR_JUDGE = True` and a `private_names.txt` file, the listed names are replaced with `[NAME]` first, but other personal details in the messages still go out. Set `RUN_LLM_JUDGE = False` to skip this part; the objective metric in D3 doesn't need it.

**Design.**
- Each pair is judged **twice, with A/B swapped**. A model only "wins" a conversation if both orders agree; anything else is a tie. This cancels the judge's position bias.
- Identical replies are ties and are never sent.
- The judge may answer `TIE`. Answers are parsed strictly: exactly one of A / B / TIE, otherwise "no answer".
- A shared rate limiter keeps total calls under `JUDGE_REQUESTS_PER_MINUTE`. Temporary errors (rate limits, timeouts, 5xx) are retried with backoff. **Permanent errors (bad key, payment/quota, unknown model) stop judging at once** with a clear message; nothing is marked as judged, so a later re-run picks everything up.
- Every verdict is written to disk immediately, keyed by conversation id. A disconnect loses nothing; re-running continues.

In [ ]:
JUDGE_INSTRUCTIONS = (
    "You compare two candidate replies in an Egyptian Arabic WhatsApp chat. You see the recent "
    "conversation, the REAL reply the person actually sent, and two CANDIDATE replies written by "
    "imitations of that person.\n"
    "Pick the candidate that is closer to how the real person replied HERE: tone, length, dialect, "
    "wording, and whether it responds to the conversation the way they did. Do not reward being more "
    "helpful, polite, or correct -- only being more like the real person.\n"
    "If both are equally close (or equally far), answer TIE.\n"
    "Answer with exactly one word: A, B, or TIE."
)


def anonymize(text):
    if not (ANONYMIZE_FOR_JUDGE and PRIVATE_NAMES):
        return text
    if NAME_RE_AR:
        text = NAME_RE_AR.sub(lambda m: (m.group("pre") or "") + "[NAME]", text)
    if NAME_RE_LA:
        text = NAME_RE_LA.sub("[NAME]", text)
    return text


def judge_context(record):
    turns = [m for m in record["messages"][:-1] if m["role"] != "system"][-JUDGE_CONTEXT_TURNS:]
    who = {"user": "Other person", "assistant": "Real person"}
    return "\n".join(f"{who.get(m['role'], m['role'])}: {anonymize(m['content'])}" for m in turns)


def build_judge_prompt(record, reply_a, reply_b):
    return (f"{JUDGE_INSTRUCTIONS}\n\nRECENT CONVERSATION:\n{judge_context(record)}\n\n"
            f"REAL REPLY: {anonymize(record['messages'][-1]['content'])}\n\n"
            f"CANDIDATE A: {anonymize(reply_a)}\n\nCANDIDATE B: {anonymize(reply_b)}\n\nAnswer:")


def parse_verdict(text):
    if not text:
        return None
    t = text.strip().upper()
    exact = re.fullmatch(r"[\W_]*(A|B|TIE)[\W_]*", t)
    if exact:
        return exact.group(1)
    found = set(re.findall(r"(?<![A-Z])(TIE|A|B)(?![A-Z])", t))
    return found.pop() if len(found) == 1 else None


class RateLimiter:
    def __init__(self, per_minute):
        self.interval, self.lock, self.next_time = 60.0 / per_minute, threading.Lock(), 0.0

    def wait(self):
        with self.lock:
            now = time.monotonic()
            start = max(now, self.next_time)
            self.next_time = start + self.interval
        time.sleep(max(0.0, start - now))

In [ ]:
if RUN_LLM_JUDGE:
    from openai import OpenAI
    judge_client = OpenAI(api_key=JUDGE_API_KEY, base_url=JUDGE_BASE_URL)
    limiter = RateLimiter(JUDGE_REQUESTS_PER_MINUTE)
    error_log = []
    STOP_JUDGING = threading.Event()
    FATAL_ERROR = []


class JudgeFatalError(Exception):
    pass


# 401 bad key, 402 payment/quota, 403 forbidden, 404 unknown model/endpoint: retrying can't fix these.
FATAL_STATUS = {401, 402, 403, 404}


def ask_judge(prompt, max_attempts=5):
    max_tokens = 300   # gpt-oss is a reasoning model: hidden reasoning counts against this budget
    for attempt in range(max_attempts):
        if STOP_JUDGING.is_set():
            raise JudgeFatalError(FATAL_ERROR[0] if FATAL_ERROR else "judging stopped")
        limiter.wait()
        try:
            resp = judge_client.chat.completions.create(
                model=JUDGE_MODEL, messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens, temperature=0.0, reasoning_effort="low",
            )
            content = resp.choices[0].message.content
            if not content or not content.strip():
                max_tokens = min(max_tokens * 2, 1200)     # reasoning used the whole budget -- give more room
                continue
            return parse_verdict(content), content.strip()[:200]
        except Exception as err:
            msg = f"{type(err).__name__}: {str(err)[:200]}"
            error_log.append(msg)
            status = getattr(err, "status_code", None)
            if status in FATAL_STATUS:
                if not STOP_JUDGING.is_set():
                    FATAL_ERROR.append(msg)
                    STOP_JUDGING.set()
                raise JudgeFatalError(msg)
            is_rate_limit = "429" in msg or "rate" in msg.lower()
            time.sleep(min(60, (8 if is_rate_limit else 2) * 2 ** attempt))
    return None, "no usable answer after retries"

In [ ]:
def judge_record(row):
    try:
        return _judge_record(row)
    except JudgeFatalError:
        return None          # not judged -- left for the next run, never saved as "failed"


def _judge_record(row):
    record = RECORDS_BY_ID[row["id"]]
    if normalize(row["sft"]) == normalize(row["dpo"]):
        return {"id": row["id"], "winner": "tie", "reason": "identical replies", "orders": None}
    # Order 1: SFT = A, DPO = B.  Order 2: DPO = A, SFT = B.
    v1, raw1 = ask_judge(build_judge_prompt(record, row["sft"], row["dpo"]))
    v2, raw2 = ask_judge(build_judge_prompt(record, row["dpo"], row["sft"]))
    to_model_1 = {"A": "sft", "B": "dpo", "TIE": "tie"}
    to_model_2 = {"A": "dpo", "B": "sft", "TIE": "tie"}
    pick1, pick2 = to_model_1.get(v1), to_model_2.get(v2)
    if pick1 is None or pick2 is None:
        winner, reason = "failed", "judge gave no usable answer"
    elif pick1 == pick2:
        winner, reason = pick1, "both orders agree"
    else:
        winner, reason = "tie", "orders disagree (position bias or a close call)"
    return {"id": row["id"], "winner": winner, "reason": reason,
            "orders": {"sft_first": pick1, "dpo_first": pick2}, "raw": [raw1, raw2]}


def load_judgments():
    done = {}
    if os.path.exists(JUDGMENTS_PATH):
        for l in open(JUDGMENTS_PATH, encoding="utf-8"):
            if l.strip():
                j = json.loads(l)
                done[j["id"]] = j
    return done


if RUN_LLM_JUDGE:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from tqdm.auto import tqdm

    # Fresh start for this run: a stop from an earlier attempt (e.g. a fixed billing problem) must not carry over.
    STOP_JUDGING.clear(); FATAL_ERROR.clear(); error_log.clear()

    judged = load_judgments()
    # Failed judgments are retried on the next run; everything else is final.
    todo = [row for row in df.to_dict("records") if judged.get(row["id"], {}).get("winner") in (None, "failed")]
    print(f"{len(judged)} judged already, {len(todo)} to judge "
          f"({sum(1 for r in todo if normalize(r['sft']) != normalize(r['dpo']))} need API calls)")

    write_lock = threading.Lock()
    with open(JUDGMENTS_PATH, "a", encoding="utf-8") as out_f, ThreadPoolExecutor(JUDGE_WORKERS) as pool:
        futures = [pool.submit(judge_record, row) for row in todo]
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Judging"):
            result = fut.result()
            if result is None:
                continue
            with write_lock:
                out_f.write(json.dumps(result, ensure_ascii=False) + "\n")
                out_f.flush()

    if STOP_JUDGING.is_set():
        print("\n" + "!" * 70)
        print("JUDGING STOPPED -- the API returned a permanent error:")
        print("  ", FATAL_ERROR[0])
        print("Check the key / billing / model name for", JUDGE_BASE_URL, "-- or switch provider in A2.")
        print("Verdicts saved so far are kept; re-run this cell after fixing it to judge the rest.")
        print("!" * 70)
    elif error_log:
        print(f"{len(error_log)} temporary API errors were retried. Most recent:", error_log[-1])
    n_failed = sum(1 for j in load_judgments().values() if j["winner"] == "failed")
    if n_failed:
        print(f"{n_failed} conversations have no verdict -- re-run this cell to retry just those.")

In [ ]:
JUDGE_RESULT = None
if RUN_LLM_JUDGE:
    judged = load_judgments()
    df["judge"] = df["id"].map(lambda i: judged.get(i, {}).get("winner", "missing"))
    counts = df["judge"].value_counts().to_dict()
    j_dpo, j_sft = counts.get("dpo", 0), counts.get("sft", 0)
    decided = j_dpo + j_sft
    judge_p = sign_test(j_dpo, j_sft)
    ci = wilson_ci(j_dpo, decided)

    sent = [j for j in judged.values() if j.get("orders")]
    consistent = sum(1 for j in sent if j["orders"]["sft_first"] == j["orders"]["dpo_first"])

    print("Judge verdicts:", {k: counts.get(k, 0) for k in ["dpo", "sft", "tie", "failed", "missing"]})
    if decided:
        print(f"DPO preferred in {j_dpo}/{decided} decided conversations ({j_dpo / decided:.0%}, "
              f"95% CI {ci[0]:.0%}-{ci[1]:.0%})   sign test p = {judge_p:.4f}")
    if sent:
        print(f"Judge consistency when A/B are swapped: {consistent}/{len(sent)} ({consistent / len(sent):.0%}) "
              "-- low consistency means the judge is mostly guessing on these pairs")
    JUDGE_RESULT = {"counts": counts, "dpo_wins": j_dpo, "sft_wins": j_sft, "p_value": judge_p,
                    "dpo_share_ci95": list(ci), "order_consistency": consistent / len(sent) if sent else None}

---
## Part F — Verdict, examples, and saved results

### F1. Overall verdict

Each line of evidence is classified in one of four ways: **DPO better**, **SFT better**, **no significant difference** (p ≥ 0.05), or **not run**. The overall verdict comes only from the **objective metric (D3)** and the **judge (E)**. Character overlap is reported as supporting context, because it's a weak proxy for style.

In [ ]:
def classify(dpo_count, sft_count, p):
    if p >= 0.05:
        return "no significant difference"
    return "DPO better" if dpo_count > sft_count else "SFT better"


evidence = {
    "Likelihood of real reply (D3)": classify(dpo_better, sft_better, nll_p),
    "LLM judge, both orders (E)": classify(JUDGE_RESULT["dpo_wins"], JUDGE_RESULT["sft_wins"], JUDGE_RESULT["p_value"])
                                  if JUDGE_RESULT else "not run",
    "Character overlap (supporting)": classify(ov_dpo, ov_sft, ov_p),
}
for k, v in evidence.items():
    print(f"  {k:34s} {v}")

primary = [evidence["Likelihood of real reply (D3)"], evidence["LLM judge, both orders (E)"]]
primary_run = [v for v in primary if v != "not run"]
if primary_run and all(v == "DPO better" for v in primary_run):
    VERDICT = "DPO is better than SFT on held-out conversations."
elif "SFT better" in primary_run and "DPO better" not in primary_run:
    VERDICT = "SFT is better than DPO on held-out conversations -- DPO hurt."
elif "DPO better" in primary_run and "SFT better" in primary_run:
    VERDICT = "The evidence conflicts (the metrics disagree) -- read the examples below before concluding."
elif "DPO better" in primary_run:
    VERDICT = "Partial evidence for DPO: one primary metric shows a significant gain, the other shows no difference."
else:
    VERDICT = "No significant difference between SFT and DPO on this sample."
print("\nVERDICT:", VERDICT)

### F2. Examples

Conversations where the two models differ most in the objective metric, plus the judge's verdict when it ran. Reading actual replies is part of the evaluation; numbers alone can hide strange behaviour.

In [ ]:
def show(rows, title):
    print("=" * 80 + f"\n{title}\n" + "=" * 80)
    for _, r in rows.iterrows():
        extra = f"  | judge: {r['judge']}" if "judge" in r else ""
        print(f"\nLAST MESSAGE: {r['last_user_msg'][:160]}")
        print(f"REAL: {r['real'][:200]}")
        print(f"SFT:  {r['sft'][:200]}")
        print(f"DPO:  {r['dpo'][:200]}")
        print(f"      NLL of real reply  SFT {r['sft_nll']:.2f}  DPO {r['dpo_nll']:.2f}{extra}")


different = df[~df["identical"]].assign(nll_gain=lambda d: d["sft_nll"] - d["dpo_nll"])
show(different.nlargest(4, "nll_gain"), "Where DPO helped most (real reply became more likely)")
show(different.nsmallest(4, "nll_gain"), "Where DPO hurt most (real reply became less likely)")

### F3. Save the table and summary

`eval_table.csv` holds every conversation with both replies and all scores. `eval_summary.json` holds the config, every metric, and the verdict. Both stay on Drive: they contain real messages and must not be committed.

In [ ]:
df.to_csv(TABLE_PATH, index=False, encoding="utf-8-sig")   # utf-8-sig so Excel shows Arabic correctly

summary = {
    "eval_run": EVAL_RUN_NAME, "config": CONFIG, "n_conversations": len(df),
    "identical_replies": int(df["identical"].sum()),
    "likelihood_of_real_reply": NLL_RESULT, "char_overlap": OVERLAP_RESULT,
    "automatic_checks": CHECKS, "llm_judge": JUDGE_RESULT,
    "judge": {"model": JUDGE_MODEL, "base_url": JUDGE_BASE_URL, "tag": JUDGE_TAG} if RUN_LLM_JUDGE else None,
    "evidence": evidence, "verdict": VERDICT,
}
json.dump(summary, open(SUMMARY_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2, default=float)
print("Saved:", TABLE_PATH)
print("Saved:", SUMMARY_PATH)

---
## Known limitations
- **Sample size.** 150 conversations detect medium effects; a small real effect can still read as "no significant difference". The effect measured during DPO training was small (+2.6 points), so expect modest numbers here too.
- **One judge.** `gpt-oss-120b` may not fully grasp Egyptian dialect nuance. The swap-consistency rate in Part E shows how often it's effectively guessing.
- **Character overlap is a weak proxy.** Two good replies can share few characters. It's reported for context, not used for the verdict.
- **Privacy.** Outputs, `eval_table.csv`, and `eval_summary.json` contain real messages. Keep them on Drive; clear notebook outputs before committing.